# Live Demo — Auditable Wikipedia RAG-ETL Pipeline

This notebook uses a **40-doc subset** but runs **identical code and schema** as the full pipeline.

---

## Pipeline Layer Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    STAGING LAYER                            │
│                                                             │
│   INGEST ──► raw_documents                                  │
│              (one row per raw doc per run, never modified)  │
│                    │                                        │
│                    ▼                                        │
│   CLEAN  ──► rejected_documents  (bad docs: EMPTY, MISSING) │
│              valid staging docs  (in-memory, passed down)   │
│                                                             │
└─────────────────────┬───────────────────────────────────────┘
                      │  validated docs
┌─────────────────────▼───────────────────────────────────────┐
│                   CURATION LAYER                            │
│                                                             │
│   CURATE ──► clean_documents  (SCD Type 2: NEW/UPDATED/UNC) │
│                    │                                        │
│                    ▼  (new/updated docs only)               │
│   CHUNK  ──► fact_chunks      (sliding window, 384 tokens)  │
│                    │                                        │
│                    ▼                                        │
│   EMBED  ──► fact_embeddings  (E5-base-v2, normalised)      │
│              + run_{id}.npy   (actual float32 vectors)      │
│                    │                                        │
│                    ▼  (all active embeddings)               │
│   INDEX  ──► faiss_index_registry  (IVFFlat)                │
│              + index.faiss + vector_map.npy                 │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

**Update contract:**  
- New id → NEW  
- Same id + same hash → UNCHANGED (idempotent)  
- Same id + different hash → UPDATED (SCD deactivates old row)  
- Empty text → REJECTED

## Setup

In [ ]:
import os, sys, subprocess, json
import duckdb
import pandas as pd

os.chdir('..')
sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

DEMO_DB     = 'outputs/live_demo/wiki_demo.duckdb'
DEMO_OUT    = 'outputs/live_demo'
DEMO_INDEX  = 'outputs/live_demo/faiss/index.faiss'

PYTHON = sys.executable
print(f'Python: {PYTHON}')

## Data Preview

In [ ]:
initial_path = 'data/live_demo/initial_sample.jsonl'
update_path  = 'data/live_demo/update_sample.jsonl'

for path in [initial_path, update_path]:
    if os.path.exists(path):
        with open(path) as f:
            docs = [json.loads(l) for l in f if l.strip()]
        print(f'{path}: {len(docs)} docs')
        for d in docs[:3]:
            print(f'  [{d["id"]}] {d["title"][:60]}')
    else:
        print(f'MISSING: {path}')

with open(initial_path) as f:
    initial_docs = [json.loads(l) for l in f if l.strip()]
REVISED_TITLE = initial_docs[0]['title']
print(f'\nRevised doc title: "{REVISED_TITLE}"')

---
# STEP 1 — Full Rebuild (first time)

Runs all 6 pipeline stages on the 40-doc initial sample.  
Expected: **40 NEW docs**, all chunks and embeddings created fresh.

In [ ]:
subprocess.run([
    PYTHON, 'src/build.py',
    '--mode', 'full',
    '--dataset-name', 'live_demo',
    '--input', 'data/live_demo/initial_sample.jsonl',
    '--output', DEMO_OUT,
    '--db', DEMO_DB,
    '--index-type', 'ivf'
], check=True)

---
## ▶ STAGING LAYER — Inspect after Step 1

The staging layer is everything produced by **INGEST** and **CLEAN**:
- `raw_documents` — verbatim copy of every ingested row
- `rejected_documents` — rows that failed validation

In [ ]:
conn = duckdb.connect(DEMO_DB, read_only=True)

print('=== STAGING: raw_documents (first 5 rows) ===')
display(conn.execute("""
    SELECT run_id, id, title, LENGTH(text) AS text_len, ingested_at
    FROM raw_documents
    ORDER BY run_id, rowid
    LIMIT 5
""").df())

raw_count = conn.execute('SELECT COUNT(*) FROM raw_documents').fetchone()[0]
print(f'\nTotal raw_documents rows: {raw_count}')

print('\n=== STAGING: rejected_documents ===')
rej = conn.execute("""
    SELECT run_id, doc_id, title, reason_code, reason_detail
    FROM rejected_documents
""").df()
display(rej)
print(f'Total rejected: {len(rej)}')

conn.close()

---
## ▶ CURATION LAYER — Inspect after Step 1

The curation layer is everything produced by **CURATE**, **CHUNK**, **EMBED**, and **INDEX**:
- `clean_documents` — active, deduplicated, versioned document records
- `fact_chunks` — sliding-window token chunks from each active document
- `fact_embeddings` — embedding metadata (vectors on disk in `.npy`)
- `faiss_index_registry` — FAISS index provenance

In [ ]:
conn = duckdb.connect(DEMO_DB, read_only=True)

print('=== CURATION: clean_documents (active, first 5) ===')
display(conn.execute("""
    SELECT doc_id, title, content_hash[:12] AS hash_prefix,
           text_len, is_active, valid_from_run_id, valid_to_run_id
    FROM clean_documents
    WHERE is_active=TRUE
    ORDER BY doc_id
    LIMIT 5
""").df())

n_active_docs = conn.execute('SELECT COUNT(*) FROM clean_documents WHERE is_active=TRUE').fetchone()[0]
print(f'Active clean_documents: {n_active_docs}')

print('\n=== CURATION: fact_chunks (sample) ===')
display(conn.execute("""
    SELECT chunk_id[:12] AS chunk_id, doc_id, chunk_index,
           start_tok, end_tok, chunk_token_len,
           LEFT(chunk_text, 60) AS chunk_preview,
           is_active
    FROM fact_chunks
    WHERE is_active=TRUE
    ORDER BY doc_id, chunk_index
    LIMIT 5
""").df())

n_chunks = conn.execute('SELECT COUNT(*) FROM fact_chunks WHERE is_active=TRUE').fetchone()[0]
print(f'Active fact_chunks: {n_chunks}')

print('\n=== CURATION: fact_embeddings (sample) ===')
display(conn.execute("""
    SELECT chunk_id[:12] AS chunk_id, embedding_model,
           embedding_dim, vector_id, source_run_id, is_active
    FROM fact_embeddings
    WHERE is_active=TRUE
    LIMIT 5
""").df())

n_emb = conn.execute('SELECT COUNT(*) FROM fact_embeddings WHERE is_active=TRUE').fetchone()[0]
print(f'Active fact_embeddings: {n_emb}')

print('\n=== CURATION: faiss_index_registry ===')
display(conn.execute("""
    SELECT index_id[:12] AS index_id, run_id, index_type,
           nlist, nprobe, num_vectors,
           build_mode, is_active, created_at
    FROM faiss_index_registry
""").df())

conn.close()

---
## Row Count Reconciliation — Step 1

Verifies: `active_chunks == active_embeddings == index_vectors`

In [ ]:
conn = duckdb.connect(DEMO_DB, read_only=True)
display(conn.execute('SELECT * FROM row_count_reconciliation ORDER BY run_id').df())
conn.close()

---
## Retrieval — Before Update

In [ ]:
from src.retrieve import retrieve

QUERIES = [
    f'What is {REVISED_TITLE}?',
    'What document discusses vector databases and retrieval systems?'
]

print('=== RETRIEVAL BEFORE UPDATE ===')
for q in QUERIES:
    print(f'\nQuery: {q}')
    df = retrieve(q, DEMO_DB, DEMO_INDEX, top_k=3)
    for _, row in df.iterrows():
        print(f'  [{row["rank"]}] {row["title"]} | score={row["score"]:.4f}')

---
# STEP 2 — Full Rebuild Again (Idempotency Test)

**Same input, same command.**  
Expected: **40 UNCHANGED** docs, 0 new chunks/embeddings, audit all PASS.

In [ ]:
subprocess.run([
    PYTHON, 'src/build.py',
    '--mode', 'full',
    '--dataset-name', 'live_demo',
    '--input', 'data/live_demo/initial_sample.jsonl',
    '--output', DEMO_OUT,
    '--db', DEMO_DB,
    '--index-type', 'ivf'
], check=True)

### Idempotency Audit Results

5 checks comparing Run 1 vs Run 2: doc count, chunk count, embedding count, index vector count, chunk-hash checksum.

In [ ]:
conn = duckdb.connect(DEMO_DB, read_only=True)
df_audit = conn.execute("""
    SELECT run_id_a, run_id_b, audit_type, result, details
    FROM audit_results
    ORDER BY created_at DESC
""").df()
display(df_audit)

passes = (df_audit['result'] == 'PASS').sum()
fails  = (df_audit['result'] == 'FAIL').sum()
print(f'\n>>> {passes} PASS  |  {fails} FAIL')
conn.close()

---
# STEP 3 — Augmented Build

Input `update_sample.jsonl` contains 4 documents with all failure modes:

| Doc | Expected outcome |
|-----|------------------|
| New doc (unseen id) | NEW → chunked & embedded |
| Revised doc (same id, different text) | UPDATED → old chunks/embeddings deactivated, new created |
| Duplicate doc (same id, same text) | UNCHANGED → nothing written |
| Corrupted doc (empty text) | REJECTED → written to rejected_documents |

In [ ]:
subprocess.run([
    PYTHON, 'src/build.py',
    '--mode', 'augmented',
    '--dataset-name', 'live_demo',
    '--input', 'data/live_demo/update_sample.jsonl',
    '--output', DEMO_OUT,
    '--db', DEMO_DB,
    '--index-type', 'ivf'
], check=True)

---
## ▶ STAGING LAYER — Inspect after Augmented Build

In [ ]:
conn = duckdb.connect(DEMO_DB, read_only=True)

print('=== STAGING: raw_documents per run ===')
display(conn.execute("""
    SELECT run_id, COUNT(*) AS ingested_rows
    FROM raw_documents
    GROUP BY run_id
    ORDER BY run_id
""").df())

print('\n=== STAGING: rejected_documents (all runs) ===')
display(conn.execute("""
    SELECT run_id, doc_id, title, reason_code, reason_detail
    FROM rejected_documents
    ORDER BY run_id
""").df())

conn.close()

---
## ▶ CURATION LAYER — Inspect after Augmented Build

The SCD logic shows the full version history.  
The revised document will have **two rows** in `clean_documents`: old (is_active=FALSE) and new (is_active=TRUE).

In [ ]:
conn = duckdb.connect(DEMO_DB, read_only=True)

print('=== CURATION: clean_documents — all versions of the revised doc ===')
display(conn.execute(f"""
    SELECT doc_id, title, content_hash[:12] AS hash_prefix,
           source_run_id, is_active, valid_from_run_id, valid_to_run_id
    FROM clean_documents
    WHERE title = '{REVISED_TITLE}'
    ORDER BY valid_from_run_id
""").df())

n_active_docs2 = conn.execute('SELECT COUNT(*) FROM clean_documents WHERE is_active=TRUE').fetchone()[0]
n_total_docs2  = conn.execute('SELECT COUNT(*) FROM clean_documents').fetchone()[0]
print(f'\nclean_documents: {n_active_docs2} active / {n_total_docs2} total (includes deactivated versions)')

print('\n=== CURATION: fact_chunks — active vs inactive for revised doc ===')
display(conn.execute(f"""
    SELECT fc.chunk_id[:12] AS chunk_id, fc.chunk_index,
           fc.source_run_id, fc.is_active,
           fc.valid_from_run_id, fc.valid_to_run_id
    FROM fact_chunks fc
    JOIN clean_documents cd ON fc.doc_id = cd.doc_id
    WHERE cd.title = '{REVISED_TITLE}'
    ORDER BY fc.source_run_id, fc.chunk_index
""").df())

n_active_chunks2 = conn.execute('SELECT COUNT(*) FROM fact_chunks WHERE is_active=TRUE').fetchone()[0]
n_total_chunks2  = conn.execute('SELECT COUNT(*) FROM fact_chunks').fetchone()[0]
print(f'\nfact_chunks: {n_active_chunks2} active / {n_total_chunks2} total')

n_active_emb2 = conn.execute('SELECT COUNT(*) FROM fact_embeddings WHERE is_active=TRUE').fetchone()[0]
print(f'fact_embeddings: {n_active_emb2} active')

print('\n=== CURATION: faiss_index_registry (all entries) ===')
display(conn.execute("""
    SELECT run_id, nlist, num_vectors, build_mode, is_active, created_at
    FROM faiss_index_registry
    ORDER BY created_at
""").df())

conn.close()

---
## Runs Table — All 3 Runs

In [ ]:
conn = duckdb.connect(DEMO_DB, read_only=True)
display(conn.execute("""
    SELECT run_id, mode, status,
           raw_doc_count,
           new_doc_count, updated_doc_count,
           unchanged_doc_count, rejected_doc_count,
           active_doc_count,
           new_chunk_count, active_chunk_count,
           new_embedding_count, active_embedding_count,
           index_vector_count
    FROM runs
    ORDER BY run_id
""").df())
conn.close()

---
## Row Count Reconciliation — All Runs

In [ ]:
conn = duckdb.connect(DEMO_DB, read_only=True)
display(conn.execute('SELECT * FROM row_count_reconciliation ORDER BY run_id').df())
conn.close()

---
## Retrieval — After Update

The second query should now surface the **revised document**, which had  
"*vector databases and retrieval systems*" appended to its text.

In [ ]:
from src.retrieve import retrieve

print('=== RETRIEVAL AFTER UPDATE ===')
for q in QUERIES:
    print(f'\nQuery: {q}')
    df = retrieve(q, DEMO_DB, DEMO_INDEX, top_k=3)
    for _, row in df.iterrows():
        print(f'  [{row["rank"]}] {row["title"]} | score={row["score"]:.4f}')
        print(f'       {str(row["chunk_text"])[:120]}')

---
## Latency Logs

In [ ]:
conn = duckdb.connect(DEMO_DB, read_only=True)
df_lat = conn.execute("""
    SELECT run_id, stage_name, duration_seconds, row_count
    FROM latency_logs
    ORDER BY run_id, start_time
""").df()
display(df_lat)
print('\nTotal seconds by run:')
display(df_lat.groupby('run_id')['duration_seconds'].sum().reset_index())
conn.close()

---
## Final Summary

| Step | Action | Key evidence |
|------|--------|--------------|
| 1 | Full build (40 docs) | `new_doc_count=40`, staging rows visible in `raw_documents` |
| 2 | Full rebuild (idempotency) | `unchanged_doc_count=40`, audit 5/5 PASS |
| 3 | Augmented build | 1 NEW, 1 UPDATED, 1 UNCHANGED, 1 REJECTED; SCD history visible |
| Retrieval | Before vs after | "vector databases" query returns revised doc post-update |

**Staging layer** (`raw_documents`, `rejected_documents`) is append-only — every run's ingestion is preserved.  
**Curation layer** (`clean_documents`, `fact_chunks`, `fact_embeddings`) uses SCD Type 2 — full version history, only active rows served.